In [ ]:
!pip install pyomo
!pip install cplex -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 MB 21.5 MB/s eta 0:00:00


**Problem Statement**

Gandhi Cloth Company is capable of manufacturing three types of clothing: shirts, shorts, and pants. The manufacture of each type of clothing requires that Gandhi have the appropriate type of machinery available. The machinery needed to manufacture each type of clothing must be rented at the following rates: shirt machinery, $\$200$ per week; shorts machinery, $\$150$ per week; pants machinery, $\$100$ per week. The manufacture of each type of clothing also requires the amounts of cloth and labor given in the table below. Each week, $150\text{ hours}$ of labor and $160\text{ sq yd}$ of cloth are available. The variable unit cost and selling price for each type of clothing are shown below. Formulate an IP whose solution will maximize Gandhi's weekly profits.

**Data Table:**

| Type | Sales Price (&#36;) | Cost (&#36;) | Labor (hours) | Cloth (sq yd) |
| --- | --- | --- | --- | --- |
| **Shirt** | 12 | 6 | 3 | 4 |
| **Shorts** | 8 | 4 | 2 | 3 |
| **Pants** | 15 | 8 | 6 | 4 |
---

**Integer Programming (IP) Formulation**

**Decision Variables**

Let:

* $x_1 =$ number of shirts produced per week ($x_1 \ge 0$, integer)
* $x_2 =$ number of shorts produced per week ($x_2 \ge 0$, integer)
* $x_3 =$ number of pants produced per week ($x_3 \ge 0$, integer)
* $y_1 = \begin{cases} 1 & \text{if shirt machinery is rented} \\ 0 & \text{otherwise} \end{cases}$
* $y_2 = \begin{cases} 1 & \text{if shorts machinery is rented} \\ 0 & \text{otherwise} \end{cases}$
* $y_3 = \begin{cases} 1 & \text{if pants machinery is rented} \\ 0 & \text{otherwise} \end{cases}$

**Profit per Unit Calculation**

* **Shirt profit:** $\$12 - \$6 = \$6$
* **Shorts profit:** $\$8 - \$4 = \$4$
* **Pants profit:** $\$15 - \$8 = \$7$

**Objective Function**

Maximize total weekly profit (Revenue $-$ Variable Costs $-$ Fixed Machinery Costs):

$$\text{Maximize } Z = 6x_1 + 4x_2 + 7x_3 - 200y_1 - 150y_2 - 100y_3$$

**Constraints**

1. **Labor Limit:**

$$3x_1 + 2x_2 + 6x_3 \le 150$$


2. **Cloth Limit:**

$$4x_1 + 3x_2 + 4x_3 \le 160$$


3. **Machinery Linkage Constraints (Big-$M$):**
* Shirt production setup: $x_1 \le M_1 y_1 \implies x_1 \le 40 y_1$ (since $\min\{150/3, 160/4\} = 40$)
* Shorts production setup: $x_2 \le M_2 y_2 \implies x_2 \le 53 y_2$ (since $\min\{150/2, 160/3\} \approx 53.33$)
* Pants production setup: $x_3 \le M_3 y_3 \implies x_3 \le 25 y_3$ (since $\min\{150/6, 160/4\} = 25$)


4. **Sign and Integrality Constraints:**

$$x_1, x_2, x_3 \ge 0 \quad \text{and integer}$$


$$y_1, y_2, y_3 \in \{0, 1\}$$

In [ ]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

In [ ]:
from pyomo.core.base import initializer
model = pyo.ConcreteModel()

# Sets
model.i = pyo.Set(initialize=['Shirt', 'Shorts', 'Pants'])

# parameters
model.Price = pyo.Param(model.i, initialize={'Shirt': 12, 'Shorts': 8, 'Pants': 15})
P = model.Price

model.Cost = pyo.Param(model.i, initialize={'Shirt': 6, 'Shorts': 4, 'Pants': 8})
Co = model.Cost

model.Fixed = pyo.Param(model.i, initialize={'Shirt': 200, 'Shorts': 150, 'Pants': 100})
F = model.Fixed

model.Labor = pyo.Param(model.i, initialize={'Shirt': 3, 'Shorts': 2, 'Pants':6})
L = model.Labor

model.Cloth = pyo.Param(model.i, initialize={'Shirt': 4, 'Shorts': 3, 'Pants': 4})
C = model.Cloth

model.M = pyo.Param(model.i, initialize={'Shirt': 40, 'Shorts': 53, 'Pants': 25})
M = model.M

# Decision variables
model.x = pyo.Var(model.i, domain=pyo.NonNegativeIntegers)
x = model.x

model.y = pyo.Var(model.i, domain=pyo.Binary)
y=model.y


#Objective Function
def Objective_rule(model,i):
  return sum(P[i]*x[i] for i in model.i) - sum(Co[i]*x[i] for i in model.i) - sum(F[i]*y[i] for i in model.i)

model.obj = pyo.Objective(rule=Objective_rule, sense=pyo.maximize)

def constraint_1(model, i):
  return sum(L[i]*model.x[i] for i in model.i) <= 150
model.con1 = pyo.Constraint(rule=constraint_1)

def constraint2(model, i):
  return sum(C[i]*model.x[i] for i in model.i) <= 160
model.con2 = pyo.Constraint(rule=constraint2)

def constraint3(model, i):
  return model.x[i] <= model.M[i]*model.y[i]
model.con3 = pyo.Constraint(model.i, rule=constraint3)

opt = SolverFactory('cplex_direct')
results = opt.solve(model, tee=True)

# print(results)
print(f'Objective Function = {model.obj()}')
for i in model.i:
  print(f"Number of {i} Produced Each Week {model.x[i].value}, y[{i}] = {model.y[i].value}")

Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 5 rows, 6 columns, and 12 nonzeros.
Reduced MIP has 3 binaries, 3 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 5 rows, 6 columns, and 12 nonzeros.
Reduced MIP has 3 binaries, 3 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 2 threads.
Root relaxation solution time = 0.00 sec. (0.01 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                            0.0000      627.0

In [ ]:
for i in model.i:
  print(f"{P[i]*model.x[i] - Co[i]*model.x[i] - F[i]*y[i]}")

12*x[Shirt] - 6*x[Shirt] - 200*y[Shirt]
8*x[Shorts] - 4*x[Shorts] - 150*y[Shorts]
x[Pants] - 8*x[Pants] - 100*y[Pants]


In [ ]:
for i in model.x:
  print(model.x[i]) # outputs symbolic representation.
  # print(type(model.x[i]))

x[Shirt]
x[Shorts]
x[Pants]
